# Historical coursework archive

This is the original 2023–2024 coursework source. Saved outputs and personal/environment metadata have been removed. **The original scores are affected by data leakage.** The source retains those issues and the differences between preparation code and supplied data. Use the revised workflow in `src/` to run the current benchmark.

Use `../src/evaluate.py`, `../notebooks/project_walkthrough.ipynb`, and `../docs/REVIEW.md` for the corrected 2026 workflow. The original code and narrative below have not been silently rewritten.


In [ ]:
# General Data Handling and Visualization
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%pip install lightgbm
%pip install catboost
%pip install xgboost
%pip install imbalanced-learn
# Sklearn - Model Selection and Evaluation
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, train_test_split, 
    cross_val_score, KFold, cross_validate
)
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error, 
    classification_report, accuracy_score
)

# Sklearn - Preprocessing
from sklearn.preprocessing import StandardScaler

# Sklearn - Ensemble Methods
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, StackingRegressor,
    RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, 
    BaggingClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
)

# Sklearn - Other Classifiers and Regressors
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Other Machine Learning Libraries
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier

# Statistical Distributions for Randomized Search
from scipy.stats import uniform, randint

from imblearn.over_sampling import SMOTE


## 1. HR Analytics

### 1.1 Load and Sample the data

#### Loading Dataset

In [ ]:
# Load the dataset
hr_data = pd.read_csv('hr_data_new.csv')

#### Additional Analysis

In [ ]:
# Check for missing values
missing_values = hr_data.isnull().sum()

# Check for unique values in categorical columns
unique_department = hr_data['department'].nunique()
unique_region = hr_data['region'].nunique()
unique_recruitment_channel = hr_data['recruitment_channel'].nunique()

# Check the balance of the target variable
promotion_counts = hr_data['is_promoted'].value_counts(normalize=True)

# Output the findings
print("Missing Values:\n", missing_values)
print("\nUnique Values:\nDepartment: {}\nRegion: {}\nRecruitment Channel: {}".format(unique_department, unique_region, unique_recruitment_channel))
print("\nPromotion Counts (Normalized):\n", promotion_counts)

* No missing values found across all columns, indicating a complete and clean dataset.
* Unique value analysis reveals moderate diversity in departments (9 types), extensive geographical coverage (34 regions), and three distinct recruitment channels.
* The target variable 'is_promoted' shows an equal split between classes, which is ideal for unbiased model training but atypical for real-world datasets.

In [ ]:
# Descriptive Statistics
print("Descriptive Statistics:\n", hr_data.describe())

In [ ]:
# Correlation Analysis
correlation_matrix = hr_data.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

### 1.2 Build the Model(s)

#### Train-Test Split

In [ ]:

# drop feature and define X and y
X = hr_data.drop('is_promoted', axis=1)
y = hr_data['is_promoted']

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25)


In [ ]:
# Apply SMOTE oversampling after train test split
X_train, y_train = SMOTE().fit_resample(X, y)

#### Train & Evaluation Function (Classifier)

In [ ]:
# Train and evaluate each base model
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)
    train_accuracy = accuracy_score(y_train, train_predictions)
    test_accuracy = accuracy_score(y_test, test_predictions)
    print(f"{model_name} Training Accuracy:", train_accuracy)
    print(f"{model_name} Test Accuracy:", test_accuracy)
    print("Classification Report on Training Data:\n", classification_report(y_train, train_predictions))
    print("Classification Report on Test Data:\n", classification_report(y_test, test_predictions))
    return model

###### This function, `evaluate_model`, is designed to fit a machine learning model to the training data, make predictions on both the training and testing datasets, and evaluate the model's performance. It outputs the training and testing accuracy to assess overfitting and underfitting and provides classification reports for a detailed performance analysis on both datasets, aiding in the model evaluation process.


#### K-Fold Cross Validation Funtion (Classifier)

In [ ]:
def evaluate_model_with_cross_validation(model, X, y, model_name, folds=5):
    # Perform cross-validation and return training and test scores
    results = cross_validate(model, X, y, cv=folds, scoring='accuracy', return_train_score=True)
    
    # Extract training and test scores
    train_scores = results['train_score']
    test_scores = results['test_score']

    # Print the training and test scores for each fold
    print(f"{model_name} Training Scores per fold: {train_scores}")
    print(f"{model_name} Test Scores per fold: {test_scores}")

    # Print the average training and test score
    print(f"{model_name} Average Training Score: {train_scores.mean():.4f}")
    print(f"{model_name} Average Test Score: {test_scores.mean():.4f}")
    print(f"{model_name} Standard Deviation of Test Scores: {test_scores.std():.4f}")

###### The `evaluate_model_with_cross_validation` function performs K-Fold Cross Validation on a given classifier. It's used to assess the model's ability to generalize to an independent dataset and mitigate overfitting. The function returns the training and test scores across the folds, calculates their mean/average and standard deviation, providing a robust measure of model performance and aids evaluation.


#### Random Forest Classifier

In [ ]:
random_forest_model = evaluate_model(RandomForestClassifier(random_state=42), X_train, y_train, X_test, y_test, "Random Forest",
)

In [ ]:
evaluate_model_with_cross_validation(RandomForestClassifier(), X, y, "Random Forest")

###### The Random Forest classifier demonstrates exceptional accuracy, achieving a training accuracy of 97.68% and a test accuracy of 95.92%. The classification reports reveal high precision, recall, and F1-scores for both classes in training and testing phases, underscoring the model's effective performance in identifying both promoted and not promoted instances.

###### Cross-validation underscores the model's consistency with an average training score of 97.34% and an average test score of 92.58%, complemented by a notably low standard deviation of test scores at 0.0019. This minuscule variation across different folds indicates the model's stable performance across diverse data subsets, showcasing its reliability.

###### The impressive metrics across the board, however, come with a cautionary note on overfitting, evident from the disparity between training and test accuracies. This observation suggests the model might be too closely fitted to the training data, a common challenge with Random Forest models given their complexity. Future efforts in model refinement should therefore prioritize hyperparameter adjustments aimed at balancing the model's fit to reduce overfitting, thereby sustaining its high accuracy in unseen data predictions.


#### XGBoost Classifier

In [ ]:
xgb_model = evaluate_model(XGBClassifier(), X_train, y_train, X_test, y_test, "XGBoost")

In [ ]:
evaluate_model_with_cross_validation(XGBClassifier(), X, y, "XGBoost")

###### The XGBoost classifier showcases robust predictive power, with a training accuracy of 92.48% and a test accuracy of 90.59%. The classification reports highlight a nuanced model performance, presenting precision, recall, and F1-scores that demonstrate the model's balanced efficacy in classifying both promoted and not promoted instances, albeit with a notable variance in precision and recall rates for the minority class in test data.


###### Cross-validation metrics reinforce the model's stability, revealing an average training score of 94.74% and an average test score of 94.15%, alongside a remarkably low standard deviation of test scores at 0.0012. This indicates a consistent model performance across various data segments, attesting to its reliability and generalizability.

###### The difference between training and test accuracies, while present, remains within a reasonable margin, suggesting a well-generalizing model with limited overfitting. The high performance on both fronts indicates the model's strong predictive capabilities, though the discrepancy in performance metrics between classes on test data suggests room for further optimization, particularly in improving the model's sensitivity and specificity across different classes.


#### Extra Trees Classifier

In [ ]:
extra_trees_model = evaluate_model(ExtraTreesClassifier(), X_train, y_train, X_test, y_test, "Extra Trees")

In [ ]:
evaluate_model_with_cross_validation(ExtraTreesClassifier(), X, y, "Extra Trees")

###### The Extra Trees classifier demonstrates impressive accuracy, with training accuracy reaching 97.68% and test accuracy at 95.93%. The classification reports for both training and test datasets reveal high precision, recall, and F1-scores, showcasing the model's adeptness at accurately classifying both promoted and not promoted instances with a slight decrease in performance for the minority class on test data.

###### Cross-validation results highlight the model's consistency, with an average training score of 97.34% and an average test score of 92.46%, alongside a low standard deviation of test scores at 0.0016. This confirms the model's stable performance across various data splits, indicating its robustness.

###### While the model exhibits a high degree of predictive accuracy, the difference between training and test accuracies suggests a potential for overfitting, albeit minor. The nuanced performance metrics, especially in the test phase, point towards a highly effective model with room for further fine-tuning. Adjusting hyperparameters and exploring feature selection could further optimize the model's performance, reducing the slight overfitting and improving its generalization capabilities on unseen data.

#### Bagging Classifier

In [ ]:
bagging_model = evaluate_model(BaggingClassifier(), X_train, y_train, X_test, y_test, "Bagging")

In [ ]:
evaluate_model_with_cross_validation(BaggingClassifier(), X, y, "Bagging")

###### The Bagging classifier exhibits outstanding performance with a training accuracy of 97.39% and a test accuracy of 95.57%. The precision, recall, and F1-scores detailed in the classification reports for both training and testing phases suggest that the model effectively discriminates between promoted and not promoted instances. Despite a slight variance in performance for the minority class in the test data, the overall metrics underscore a model capable of maintaining high accuracy across different classes.

###### The cross-validation analysis underscores the model's consistency, with an average training score of 96.92% and an average test score of 92.37%, coupled with a standard deviation of test scores at 0.0018. These figures indicate a stable performance across various folds, highlighting the model's robustness and its ability to generalize well to new data.

###### This model is considered to have minor to no overfitting. It combines high predictive power with a commendable ability to generalize, making it a potent tool for predictive tasks. Though the model is already performing at a high level, exploring further adjustments in hyperparameters and feature engineering could provide marginal gains, especially in enhancing its precision and recall for the less represented class.


#### KNN Classifier

In [ ]:
knn_model = evaluate_model(KNeighborsClassifier(), X_train, y_train, X_test, y_test, "KNN")

In [ ]:
evaluate_model_with_cross_validation(KNeighborsClassifier(), X, y, "KNN")

###### The KNN classifier demonstrates excellent accuracy, with training accuracy reaching 91.51% and test accuracy slightly higher at 91.79%. This unusual case where test accuracy surpasses training accuracy can occur due to the specific characteristics of the test data or the model's ability to generalize exceptionally well to new data. The classification reports highlight a solid performance across both classes in the training phase and an interesting dynamic in the test phase, where precision and recall vary significantly between classes, indicating differential effectiveness in classifying instances.

###### The cross-validation metrics further attest to the model's robustness, with an average training score of 93.64% and an average test score of 92.29%, alongside a very low standard deviation of test scores at 0.0013. Such consistency and low variability suggest the model performs reliably across different data subsets, demonstrating its stability and generalizability.

###### The KNN model's performance metrics suggest it is a powerful tool for predictive tasks, capable of maintaining high accuracy across diverse datasets. While the precision and recall scores for the minority class in test data indicate room for improvement, the overall high accuracy and consistency across folds signify a well-tuned model. Future enhancements could explore strategies for balancing class performance, potentially through advanced sampling techniques or parameter tuning to further refine its predictive accuracy.


#### LightGBM Classifier

In [ ]:
light_gbm_model = evaluate_model(LGBMClassifier(), X_train, y_train, X_test, y_test, "Light GBM")

In [ ]:
evaluate_model_with_cross_validation(LGBMClassifier(), X, y, "Light GBM")

###### The LightGBM model showcases impressive accuracy, with a training accuracy of 93.25% and a test accuracy of 91.49%. This model demonstrates strong generalization capabilities, with a relatively small gap between training and test accuracies. The classification reports for both phases of evaluation reveal that the model performs exceptionally well for the majority class, achieving high precision, recall, and F1-scores. However, there is a notable disparity in performance when it comes to the minority class, as evidenced by lower precision and recall scores in the test data.

###### The cross-validation analysis affirms the model's consistent and reliable performance, with an average test score of 94.24% that surprisingly exceeds the reported single test accuracy, suggesting exceptionally stable performance across different validation folds. The standard deviation of test scores at 0.0013 indicates minimal variability in the model's predictive power across these folds.

###### Overall, the LightGBM classifier emerges as a robust model with strengths in handling varied data representations effectively. Its high precision and recall for the majority class alongside stable cross-validation scores underscore its potential for real-world applications. To enhance its performance further, especially for the minority class, future work could explore more targeted approaches in feature engineering, class weighting, or advanced sampling techniques.


#### CatBoost Classifier

In [ ]:
catboost_model = evaluate_model(CatBoostClassifier(verbose=False), X_train, y_train, X_test, y_test, "CatBoost")

In [ ]:
evaluate_model_with_cross_validation(CatBoostClassifier(verbose=False), X, y, "Catboost")

###### The CatBoost model delivers exceptional performance, with a training accuracy of 95% and a test accuracy of 93.75%. The detailed classification reports underscore the model's proficiency in accurately classifying instances across both the training and test datasets, showcasing high precision, recall, and F1-scores for the majority class. Notably, there is a reduction in performance metrics for the minority class within the test data, indicating areas for targeted improvement.

###### The cross-validation results further emphasize the model's stable and reliable performance, with an average training score of 94.73% and an average test score of 94.23%. The remarkably low standard deviation of test scores at 0.0010 across folds highlights the model's consistency, underscoring its ability to perform well across various subsets of data.

###### Overall, the CatBoost classifier stands out for its robust predictive accuracy and balanced classification capabilities, especially for the majority class. While it excels in general performance, the disparity in precision and recall for the minority class in the test phase suggests potential avenues for refinement, possibly through more nuanced feature engineering or advanced ensemble strategies. This model's strong foundation and high degree of generalizability make it a compelling choice for predictive tasks, with opportunities to enhance its inclusivity and precision for all classes.


#### MLP Classifier

In [ ]:
mlp_model = evaluate_model(MLPClassifier(), X_train, y_train, X_test, y_test, "MLP")

In [ ]:
evaluate_model_with_cross_validation(MLPClassifier(), X, y, "MLP")

###### The MLP Classifier demonstrates a training accuracy of 82.49% and a notable decrease in test accuracy to 75.65%. This discrepancy suggests challenges in generalizing the model to unseen data. The classification reports reveal a high recall for the positive class in the training set, but a stark contrast in precision and recall scores between classes in the test data, indicating a significant imbalance in the model's predictive capabilities.

###### The model's cross-validation scores present an intriguing picture: with an average training score of 93.66% and an average test score of 93.48%, which starkly contrasts the single test accuracy reported. This discrepancy may be attributed to the model's sensitivity to specific data configurations or a potential misalignment between cross-validation methodology and test set evaluation. The standard deviation of test scores at 0.0017 across folds suggests high stability in the model's performance under cross-validation conditions.

###### Despite the high cross-validation scores, the substantial gap between training and test accuracies in the standalone evaluation highlights a need for further investigation and potential model tuning. The convergence warning indicates that the model has not fully optimized its parameters within the given iterations, pointing to the necessity for either increasing the number of iterations or adjusting other hyperparameters to achieve better convergence and potentially improve generalization to unseen data.

### 1.3 Evaluate and Improve the Model(s)

#### Enhanced Random Forest

In [ ]:
# Parameter grid for RandomizedSearchCV
param_dist_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

# Initialize RandomizedSearchCV
random_search_rf = RandomizedSearchCV(RandomForestClassifier(), param_distributions=param_dist_rf, n_iter=100, cv=5, verbose=2, random_state=42, n_jobs=-1
)

# Fitting RandomizedSearchCV with resampled and feature-selected data
random_search_rf.fit(X_train, y_train)

print("Best Score:", random_search_rf.best_score_)
print("Best Parameters:", random_search_rf.best_params_)

# Evaluate the best model from random search
rf_model = RandomForestClassifier(**random_search_rf.best_params_)
evaluate_model(rf_model, X_train, y_train, X_test, y_test, "Tuned Random Forest")

evaluate_model_with_cross_validation(rf_model, X, y, "Tuned Random Forest")

###### The optimized Random Forest model, after comprehensive hyperparameter tuning, exhibits exceptional performance with a training accuracy of 97.34% and a test accuracy of 96.01%. The tuning process, incorporating parameters like 'n_estimators': 300, 'max_features': 'log2', and 'min_samples_split': 10, significantly enhanced the model's predictive capabilities, as evident from the classification reports. These reports indicate nearly perfect precision, recall, and F1-scores for the majority class across both training and testing phases, with a notable precision and recall for the minority class in the test data.

###### Cross-validation analysis further demonstrates the model's robustness, yielding an average training score of 96.05% and an average test score of 93.44%, alongside a standard deviation of test scores at 0.0015. These metrics underscore the model's stable performance across different folds, highlighting its reliability and generalizability post-tuning.

###### This fine-tuned version of the Random Forest classifier not only shows a significant improvement over its default configuration but also emphasizes the value of hyperparameter tuning in achieving superior accuracy and generalization. The model's enhanced performance makes it a highly effective tool for predictive tasks, with its ability to accurately classify instances and maintain high performance metrics across diverse datasets.


#### Enhanced XGBoost Classifier

In [ ]:
# XGBoost
param_dist_xgb = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.1, 1, 10],
    'reg_lambda': [1, 0.1, 0.01, 0],
    'min_child_weight': [1, 3, 5, 7],
    'scale_pos_weight': [1, (y_train.value_counts()[0] / y_train.value_counts()[1])]
}
random_search_xgb = RandomizedSearchCV(XGBClassifier(), param_distributions=param_dist_xgb, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1)
random_search_xgb.fit(X_train, y_train)

print("Best Score:", random_search_xgb.best_score_)
print("Best Parameters:", random_search_xgb.best_params_)

xgb_model = XGBClassifier(**random_search_xgb.best_params_)
xgb_model = evaluate_model(xgb_model, X_train, y_train, X_test, y_test, "Tuned XGBoost")

evaluate_model_with_cross_validation(xgb_model, X, y, "Tuned XGBoost")


###### The Tuned XGBoost model, after hyperparameter optimization, exhibits a significant improvement in performance, achieving a training accuracy of 95.71% and a test accuracy of 93.90%. This optimization process has substantially enhanced the model's predictive capability, as reflected in the classification reports. These reports indicate a nearly uniform precision, recall, and F1-scores across both classes in the training phase, and notable improvement in the model's ability to generalize to unseen data, especially in increasing the recall for the minority class in the test set.

###### The optimal set of hyperparameters, including 'subsample': 0.9, 'scale_pos_weight': 1, 'reg_lambda': 0.1, among others, have been instrumental in boosting the model's overall accuracy and balanced performance. The cross-validation analysis supports these findings, showing an average training score of 96.31% and an average test score of 93.11%, with a very low standard deviation of test scores at 0.0017, indicating the model's enhanced stability and reliability across different data segments.

###### This tuned version of the XGBoost classifier demonstrates a marked improvement over its default configuration, showcasing the value of hyperparameter tuning in achieving higher accuracy and better generalization capabilities. The adjustments made through tuning have rendered the XGBoost model even more compelling for complex predictive tasks, highlighting its efficiency and effectiveness in handling classification challenges.


#### Enhanced Extra Trees Classifier

In [ ]:
# Tuned Extra Trees
param_grid_et = {
    'n_estimators': [100, 150],
    'max_depth': [10, 20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'bootstrap': [True]
}
grid_et = GridSearchCV(ExtraTreesClassifier(), param_grid_et, cv=3, n_jobs=-1, verbose=2)
grid_et.fit(X_train, y_train)

print("Best Score:", grid_et.best_score_)
print("Best Parameters:", grid_et.best_params_)

extra_trees_model = ExtraTreesClassifier(**grid_et.best_params_)
extra_trees_model = evaluate_model(extra_trees_model, X_train, y_train, X_test, y_test, "Tuned Extra Trees")

evaluate_model_with_cross_validation(extra_trees_model, X, y, "Tuned Extra Trees")


###### After hyperparameter tuning, the Extra Trees model shows a notable improvement, achieving a training accuracy of 89.81% and a test accuracy of 83.79%. The tuning process identified optimal parameters including 'bootstrap': True, 'max_depth': 20, and 'n_estimators': 150, which contributed to enhancing the model's performance. The classification reports highlight a significant improvement in recall for the positive class in the training data and a commendable ability to maintain high precision and recall scores in test data, especially for the minority class.

###### The model's cross-validation performance further substantiates its robustness, with an average training score of 94.08% and an average test score of 93.63%, alongside a very low standard deviation of test scores at 0.0011. These metrics indicate a substantial boost in stability and reliability across different subsets of data post-tuning.

###### This tuned version of the Extra Trees classifier thus represents a significant step forward in terms of accuracy and generalization capability, demonstrating the effectiveness of hyperparameter tuning in optimizing the model for better performance across varied data scenarios.


#### Enhanced Bagging Classifier

In [ ]:
# Tuned Bagging
param_grid_bag = {
    'n_estimators': [50, 100, 150, 200],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0],
    'bootstrap': [True],
    'bootstrap_features': [False, True]
}
grid_bag = GridSearchCV(BaggingClassifier(), param_grid_bag, cv=3, n_jobs=-1, verbose=2)
grid_bag.fit(X_train, y_train)

print("Best Score:", grid_bag.best_score_)
print("Best Parameters:", grid_bag.best_params_)

bagging_model = BaggingClassifier(**grid_bag.best_params_)
bagging_model = evaluate_model(bagging_model, X_train, y_train, X_test, y_test, "Tuned Bagging")

evaluate_model_with_cross_validation(bagging_model, X, y, "Tuned Bagging")


###### The Tuned Bagging classifier, after extensive hyperparameter optimization, showcases a substantial enhancement in its predictive performance, achieving a training accuracy of 96.42% and a test accuracy of 95.19%. The optimization identified 'bootstrap': True and 'n_estimators': 100 as part of the best parameter set, significantly improving the model's precision, recall, and F1-scores, particularly highlighting its efficiency in classifying the majority class with high precision and recall in the training phase and maintaining robust accuracy in the test phase.

###### The cross-validation metrics further underscore the model's enhanced stability and reliability, with an average training score of 94.93% and an average test score of 94.00%, accompanied by a minimal standard deviation of test scores at 0.0016. These results indicate a marked improvement in the model's consistency across different data segments and its robustness to variance in the data.

###### This tuned Bagging classifier not only demonstrates a significant leap in performance metrics over its default setup but also illustrates the impact of thoughtful hyperparameter tuning in optimizing model performance for higher accuracy and better generalization across unseen data, making it an even more effective tool for predictive analysis.


#### Enhanced KNN Classifier

In [ ]:
# Parameters for tuning KNN
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, verbose=2, n_jobs=-1, scoring='accuracy')
grid_knn.fit(X_train, y_train)

print("Best Score:", grid_knn.best_score_)
print("Best Parameters:", grid_knn.best_params_)

knn_model = KNeighborsClassifier(**grid_knn.best_params_)
knn_model = evaluate_model(knn_model, X_train, y_train, X_test, y_test, "Tuned KNN")

evaluate_model_with_cross_validation(knn_model, X, y, "Tuned KNN")

###### The Tuned KNN (K-Nearest Neighbors) model, after optimization, achieved remarkable improvements, showcasing a training accuracy of 97.00% and a test accuracy of 96.88%. The selected parameters, including 'n_neighbors': 11 and 'weights': 'distance', have significantly contributed to this enhancement. The classification reports for both training and testing phases indicate exceptional precision, recall, and F1-scores, particularly showcasing the model's strength in accurately classifying instances with nearly perfect metrics for the majority class and substantial improvement in the minority class's recall in the test data.

###### Cross-validation analysis reveals an average training score of 97.34% and an average test score of 91.84%, with a standard deviation of test scores at 0.0012. These figures highlight the model's consistency and reliability across different folds, demonstrating its robustness and stability.

###### This optimized KNN model exemplifies the profound impact of hyperparameter tuning in enhancing model performance, significantly improving both accuracy and generalization capabilities. The remarkable alignment between training and test accuracies underscores the model's efficiency in handling classification tasks, making it a potent tool for predictive modeling with high precision and recall.

#### Enhanced Light GBM Classifier

In [ ]:
import numpy as np
# LightGBM Classifier
param_dist_lgbm = {
'n_estimators': np.linspace(100, 1000, 10, dtype=int),
    'learning_rate': [0.01, 0.05, 0.001],
    'num_leaves': np.linspace(20, 60, 5, dtype=int),
    'max_depth': [5, 10, 15, -1],
    'min_child_samples': [10, 20, 30],
    'max_bin': [255, 510],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'subsample_freq': [1],
    'colsample_bytree': [0.6, 0.8, 1.0]
}
random_search_lgbm = RandomizedSearchCV(LGBMClassifier(), param_distributions=param_dist_lgbm, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1)
random_search_lgbm.fit(X_train, y_train)

print("Best Score:", random_search_lgbm.best_score_)
print("Best Parameters:", random_search_lgbm.best_params_)

light_gbm_model = LGBMClassifier(**random_search_lgbm.best_params_)
light_gbm_model = evaluate_model(light_gbm_model, X_train, y_train, X_test, y_test, "Tuned Light GBM")

evaluate_model_with_cross_validation(light_gbm_model, X, y, "Tuned Light GBM")

###### The optimized Light GBM model, following hyperparameter tuning, demonstrates significant improvements, reaching a training accuracy of 96.42% and a test accuracy of 94.74%. The tuning process, which incorporated parameters like 'num_leaves': 60, 'n_estimators': 800, and 'learning_rate': 0.05, has significantly enhanced the model's precision, recall, and F1-scores, particularly highlighting its effectiveness in accurately identifying both classes with a notable emphasis on maintaining high precision and recall for the majority class in both training and test phases.

###### The cross-validation metrics further affirm the model's robustness, with an average training score of 95.91% and an average test score of 93.77%, alongside a minimal standard deviation of test scores at 0.0009. These results underscore the model's consistent and reliable performance across different data segments, evidencing its stability and generalizability.

###### This fine-tuned version of the Light GBM classifier not only showcases a marked advancement in accuracy and generalization capabilities over its default configuration but also underscores the effectiveness of meticulous hyperparameter tuning in optimizing model performance for complex predictive tasks, thus solidifying its status as a highly reliable and efficient tool for predictive modeling.


#### Enhanced CatBoost Classifier

In [ ]:
# CatBoost Classifier
param_dist_cb = {
'iterations': np.linspace(100, 500, 5, dtype=int),
    'learning_rate': np.linspace(0.01, 0.2, 5),
    'depth': [4, 6, 8, 10],
    'l2_leaf_reg': np.logspace(-1, 1, 5),
    'border_count': [32, 64, 128, 255],
    'early_stopping_rounds': [20]  # Adding early stopping
}
random_search_cb = RandomizedSearchCV(
    CatBoostClassifier(verbose=False, task_type='CPU', eval_metric='Accuracy'),  
    param_distributions=param_dist_cb,
    n_iter=50, 
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
random_search_cb.fit(X_train, y_train)

print("Best Score:", random_search_cb.best_score_)
print("Best Parameters:", random_search_cb.best_params_)

catboost_model = CatBoostClassifier(**random_search_cb.best_params_)
catboost_model = evaluate_model(catboost_model, X_train, y_train, X_test, y_test, "Tuned CatBoost")

evaluate_model_with_cross_validation(catboost_model, X, y, "Tuned CatBoost")

###### After hyperparameter tuning, the CatBoost model exhibits substantial improvements, achieving a training accuracy of 96.08% and a test accuracy of 94.71%. The selected hyperparameters, including 'learning_rate': 0.105 and 'depth': 10, have significantly contributed to this enhancement, as evidenced by the balanced and high precision, recall, and F1-scores for both classes in the training phase, and commendable precision and recall for the majority class in the test phase.

###### The cross-validation results further underscore the model's enhanced stability and reliability, with an average training score of 95.98% and an average test score of 93.71%, accompanied by a standard deviation of test scores at 0.0016. These metrics indicate a marked improvement in the model's consistency across different data segments, demonstrating its robustness and generalizability.

###### This fine-tuned version of the CatBoost classifier not only showcases significant advancements in accuracy and generalization capabilities but also highlights the effectiveness of careful hyperparameter tuning in optimizing model performance for sophisticated predictive tasks, solidifying its status as a highly efficient and reliable tool for predictive analytics.


#### Enhanced MLP Classifier

In [ ]:
# Tuned MLP
param_dist_mlp = {
'hidden_layer_sizes': [(50, 50), (100, 50), (100, 100), (150, 100, 50)],
    'activation': ['tanh', 'relu'],
    'solver': ['sgd', 'adam'],
    'alpha': [0.0001, 0.001, 0.005],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.001, 0.01],  
    'max_iter': [500, 1000], 
    'batch_size': [64, 128, 256], 
    'momentum': [0.9, 0.95]  
}
random_search_mlp = RandomizedSearchCV(MLPClassifier(), param_distributions=param_dist_mlp, n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1)
random_search_mlp.fit(X_train, y_train)

print("Best Score:", random_search_mlp.best_score_)
print("Best Parameters:", random_search_mlp.best_params_)

mlp_model = MLPClassifier(**random_search_mlp.best_params_)
mlp_model = evaluate_model(mlp_model, X_train, y_train, X_test, y_test, "Tuned MLP")

evaluate_model_with_cross_validation(mlp_model, X, y, "Tuned MLP")

###### The Tuned MLP (Multi-Layer Perceptron) model, optimized through extensive hyperparameter tuning, achieved a training accuracy of 91.66% and a test accuracy of 87.80%. The chosen parameters, such as 'solver': 'sgd', 'learning_rate': 'adaptive', and 'hidden_layer_sizes': (150, 100, 50), have markedly enhanced the model's performance. The classification reports indicate a strong precision and recall in the training phase and a significant improvement in recall for the minority class in the test data, albeit with a decrease in precision for this class.

###### The cross-validation metrics underscore the model's improved performance and stability, with an average training score of 94.43% and an average test score of 94.23%, alongside a standard deviation of test scores at 0.0014. These results highlight the model's consistency across different folds, demonstrating its robustness and enhanced generalizability post-tuning.

###### This optimized MLP model showcases a notable advancement in its predictive capabilities, reflecting the benefits of thorough hyperparameter tuning. While the improved recall for the minority class in the test set is commendable, the precision for this class suggests an area for further improvement, potentially through additional tuning or adopting techniques to balance class representation.


### 1.4 Best Model

Among the models evaluated, the **Tuned KNN** emerges as the standout model for the classification task at hand. It achieved a training accuracy of **97.00%** and a test accuracy of **96.88%**, demonstrating its exceptional ability to accurately classify instances and maintain high predictive performance on unseen data. The precision, recall, and F1-scores across both training and test phases indicate the model's balanced performance, particularly its effectiveness in classifying both majority and minority classes with high precision and recall.

The strength of the Tuned KNN model lies in its optimized parameter set, including `'n_neighbors': 11` and `'weights': 'distance'`, which significantly contributed to its improved performance. This optimization process has effectively leveraged the KNN's inherent capabilities for classification, enhancing its precision and recall, especially for the minority class in the test data, thereby addressing common challenges such as class imbalance.

In comparison to other models evaluated, including Tuned versions of XGBoost, LightGBM, CatBoost, and Random Forest, the Tuned KNN not only achieved one of the highest accuracies but also demonstrated a remarkable balance between training and test performance. This indicates a superior generalization capability, minimizing the risk of overfitting and ensuring consistent performance across different data subsets.

Overall, the **Tuned KNN model** represents a significant achievement in classification modeling, providing a highly effective and reliable solution for predictive tasks. Its ability to deliver precise classifications across diverse datasets, combined with its robustness and adaptability, positions it as the preferred choice for achieving high accuracy in complex classification scenarios.


## 2. Airbnb

### 2.1 Load and Sample the data

In [ ]:
# Load the dataset
listings = pd.read_csv('listings_new.csv')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Set the aesthetic style of the plots
sns.set_style('whitegrid')

# Initialize the matplotlib figure
plt.figure(figsize=(10, 8))

# Plot a histogram with a kernel density estimate
# Removed kde_kws from histplot as it is not needed here
sns.histplot(listings['price'], kde=True, bins=30, color="g")

# Plotting the KDE line using sns.kdeplot for customizations
sns.kdeplot(listings['price'], color="k", lw=3)

# Set title and labels
plt.title('Skewness of Price Variable', fontsize=16)
plt.xlabel('Price', fontsize=14)
plt.ylabel('Density', fontsize=14)

# Remove the top and right spines
sns.despine(trim=True)

# Show the plot
plt.show()

* The plot illustrates the skewness in Airbnb listing prices, with a clear right-skewed distribution indicating most listings are at a lower price range.
* A histogram with a Kernel Density Estimate (KDE) overlay provides a visual representation of the price density and distribution.
* The long tail to the right suggests higher-priced listings are less frequent, which could affect the performance of regression models.

In [ ]:
import numpy as np
# Apply log transformation using numpy's log1p which is log(x + 1) to avoid log(0)
listings['price'] = np.log1p(listings['price'])

# Plotting the transformed variable
sns.histplot(listings['price'], kde=True)
plt.title('Log-transformed Price Distribution')
plt.xlabel('Log(Price + 1)')
plt.ylabel('Density')
plt.show()

* This histogram shows the distribution of Airbnb prices after log transformation, reducing the right skewness and normalizing the distribution.
* Log transformation is a common technique to manage skewed data, enhancing model accuracy.
* The more symmetric distribution around the peak suggests improved homoscedasticity, important for linear regression models.

### 2.2 Build the Model(s)

#### Train-Test Split

In [ ]:
# Assuming 'price' is the target variable and other columns are features
# Replace 'price' with the correct column name if it's different
X = listings.drop('price', axis=1)
y = listings['price']

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Train & Evaluation Function (Regressor)

In [ ]:
# Function to evaluate regression model
def evaluate_regression_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    train_mse = mean_squared_error(y_train, train_predictions)
    test_mse = mean_squared_error(y_test, test_predictions)
    train_mae = mean_absolute_error(y_train, train_predictions)
    test_mae = mean_absolute_error(y_test, test_predictions)
    train_r2 = r2_score(y_train, train_predictions)
    test_r2 = r2_score(y_test, test_predictions)

    print(f"\n{model_name} - Training MSE:", train_mse)
    print(f"{model_name} - Test MSE:", test_mse)
    print(f"{model_name} - Training MAE:", train_mae)
    print(f"{model_name} - Test MAE:", test_mae)
    print(f"{model_name} - Training R² Score:", train_r2)
    print(f"{model_name} - Test R² Score:", test_r2)

    return model

###### The `evaluate_regression_model` function fits the regression model to the training data, makes predictions, and computes performance metrics.
###### Metrics calculated include Mean Squared Error (MSE), Mean Absolute Error (MAE), Mean Absolute Percentage Error (MAPE), and R² Score for both training and test datasets.
###### Outputs are printed for each metric, providing insight into the model's accuracy and predictive power.

#### K-Fold Cross Validation Funtion (Regressor)

In [ ]:
def evaluate_regression_model_with_cross_validation(model, X, y, model_name, folds=5):
    scoring_metrics = {
        'MSE': 'neg_mean_squared_error'
    }

    # Perform cross-validation with MSE scoring metric
    results = cross_validate(model, X, y, cv=folds, scoring=scoring_metrics, return_train_score=False)

    # Process and print results for MSE
    # Extract test scores and negate the values to get the actual scores
    test_scores = -results['test_MSE']

    # Print the test scores for each fold
    print(f"{model_name} - MSE - Test Scores per fold: {test_scores}")

    # Print the average test score
    print(f"{model_name} - MSE - Average Test Score: {test_scores.mean():.4f}\n")

#### Random Forest Regressor

In [ ]:
evaluate_regression_model(RandomForestRegressor(), X_train, y_train, X_test, y_test, "Random Forest")

In [ ]:
evaluate_regression_model_with_cross_validation(RandomForestRegressor(), X, y, "Random Forest")

###### The Random Forest regression model shows considerable strength with a training R² score of 94.95%, indicating a substantial portion of the variance in the target variable is explained by the model. However, the test R² score drops to 67.99%, suggesting a decrease in predictive accuracy on unseen data. This gap between training and test performance points towards overfitting within the model.

###### Regarding error metrics, the model records a training Mean Squared Error (MSE) of 0.0286 and a Mean Absolute Error (MAE) of 0.1088, reflecting relatively small prediction errors on the training dataset. In contrast, on the test dataset, the MSE and MAE escalate to 0.1726 and 0.2814, respectively, indicating increased errors when forecasting new observations, aligning with the overfitting diagnosis.

###### Cross-validation with five folds reveals an average test MSE of 0.2465, higher than single-split test results, which underscores the model's varied performance across different data segments. The distribution of MSE scores per fold, ranging from 0.2019 to 0.3129, further illustrates this variability and the challenges in achieving consistent prediction accuracy.

###### The notable discrepancy between training and test scores, alongside the variance in cross-validation MSE, underlines the model's overfitting issue. Strategies for mitigation include tuning hyperparameters such as the number of estimators, tree maximum depth, and minimum samples split to enhance model generalization and reduce overfitting.


#### Gradient Boosting Regressor

In [ ]:
evaluate_regression_model(GradientBoostingRegressor(), X_train, y_train, X_test, y_test, "Gradient Boosting")

In [ ]:
evaluate_regression_model_with_cross_validation(GradientBoostingRegressor(), X, y, "Gradient Boosting")

###### The Gradient Boosting model exhibits a training R² score of 63.62%, demonstrating its capability to explain a modest proportion of the variance in the target variable. The test R² score slightly decreases to 62.23%, indicating a relatively consistent predictive accuracy on unseen data, which suggests minimal overfitting compared to the Random Forest model.

###### In terms of error metrics, the model achieves a training Mean Squared Error (MSE) of 0.2059 and a Mean Absolute Error (MAE) of 0.3236. These figures marginally increase in the test set to an MSE of 0.2036 and an MAE of 0.3307, reflecting a slight elevation in prediction errors when applied to new data but maintaining a close alignment between training and test performance.

###### Cross-validation across five folds presents an average test MSE of 0.2437, with individual fold scores ranging from 0.2128 to 0.2709. This variation in MSE across folds highlights the model's stability and its reasonable adaptability to different subsets of data, though with a noted increase in prediction errors in the cross-validation process.

###### The Gradient Boosting model's relatively stable performance between training and testing phases, along with its consistent cross-validation results, indicates a better balance between bias and variance. However, the overall performance metrics suggest there is room for improvement in model accuracy. Enhancements could include adjusting hyperparameters related to the learning rate, tree depth, and number of estimators to further refine the model's predictive capabilities.


#### XGBoost Regressor

In [ ]:
evaluate_regression_model(XGBRegressor(), X_train, y_train, X_test, y_test, "XGBoost")

In [ ]:
evaluate_regression_model_with_cross_validation(XGBRegressor(), X, y, "XGBoost")

###### The XGBoost regression model showcases a robust performance with a high training R² score of 92.93%, indicating its effectiveness in explaining a significant portion of the variance in the target variable. Nonetheless, the test R² score drops to 65.46%, revealing a notable decline in prediction accuracy on unseen data, which suggests potential overfitting.

###### Regarding error metrics, the training Mean Squared Error (MSE) is low at 0.04, and the Mean Absolute Error (MAE) is 0.1395, indicating precise predictions on the training set. However, these metrics worsen in the test set, with MSE increasing to 0.1862 and MAE to 0.3022, demonstrating greater prediction errors when the model is applied to new data.

###### Cross-validation results, using five folds, exhibit an average test MSE of 0.2689. The MSE scores per fold range from 0.2253 to 0.3482, highlighting the model's variability in performance across different data segments. This variability and the substantial difference between training and test results further point to overfitting issues within the model.

###### The XGBoost model, while powerful in training, shows signs of overfitting as evidenced by the significant discrepancies between its training and test performances, and its varied cross-validation MSE scores. Addressing overfitting could involve fine-tuning hyperparameters such as learning rate, tree complexity, and regularization terms. These adjustments aim to improve the model's generalization ability and ensure more consistent performance across different datasets.


#### Light GBM Regressor

In [ ]:
evaluate_regression_model(LGBMRegressor(), X_train, y_train, X_test, y_test, "LightGBM")

In [ ]:
evaluate_regression_model_with_cross_validation(LGBMRegressor(), X, y, "LightGBM")

###### The LightGBM regression model demonstrates a solid performance with a training R² score of 77.85%, indicating its capability to account for a considerable portion of the variance in the target variable. The test R² score sees a slight decrease to 68.09%, which suggests a modest drop in predictive accuracy on unseen data, potentially indicating slight overfitting but with a relatively close alignment between training and test performances.

###### In terms of error metrics, the model yields a training Mean Squared Error (MSE) of 0.1254 and a Mean Absolute Error (MAE) of 0.2420, pointing towards acceptable prediction errors on the training dataset. On the test dataset, the MSE and MAE increase to 0.1720 and 0.2953, respectively, showcasing a rise in prediction errors when the model confronts new data, yet maintaining a reasonable gap that suggests controlled overfitting.

###### Cross-validation employing five folds results in an average test MSE of 0.2358, with individual fold scores indicating some variability (ranging from 0.2019 to 0.2637) in the model's prediction errors across different data segments. This cross-validation performance supports the model's ability to generalize, albeit with some inconsistencies across folds.

###### LightGBM's performance, marked by a decent balance between training and testing accuracy and stable cross-validation results, underscores its effectiveness while hinting at areas for potential improvement. Tuning the model's hyperparameters, such as the number of leaves, min data in leaf, and learning rate, could further optimize its performance, aiming to reduce overfitting and enhance its generalization capabilities across diverse datasets.


#### CatBoost Regressor

In [ ]:
evaluate_regression_model(CatBoostRegressor(verbose=False), X_train, y_train, X_test, y_test, "CatBoost")

In [ ]:
evaluate_regression_model_with_cross_validation(CatBoostRegressor(verbose=False), X, y, "CatBoost")

###### The CatBoost regression model exhibits strong training performance with an R² score of 83.20%, showcasing its efficiency in capturing a large fraction of the variance within the target variable. The model's test R² score slightly drops to 67.69%, indicating a decent level of predictive accuracy on unseen data, with a relatively small performance gap hinting at mild overfitting.

###### Error metrics reveal a training Mean Squared Error (MSE) of 0.0951 and a Mean Absolute Error (MAE) of 0.2188, signifying precise predictions on the training data. Conversely, the test dataset experiences a slight increase in MSE to 0.1742 and in MAE to 0.2972, indicating that the model's predictions are less accurate on new data, although the increase is moderate and suggests the model is relatively well-generalized.

###### Through cross-validation with five folds, the model achieves an average test MSE of 0.2344, with MSE scores per fold showing some variation (from 0.2049 to 0.2685). This variability across folds underscores the model's consistent yet flexible performance on different segments of the data, pointing to its robustness and adaptability.

###### CatBoost's overall performance, characterized by strong training and test results alongside stable cross-validation outcomes, demonstrates its potent predictive capabilities. To further refine its accuracy and mitigate any overfitting, adjusting model hyperparameters such as depth, learning rate, and iterations, as well as exploring feature engineering techniques, could be beneficial strategies for enhancing model generalization across varied datasets.


### 2.3 Evaluate and Improve the Model(s)

#### Enhanced Random Forest Regressor

In [ ]:

# Define the parameter grid
param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20],  
    'min_samples_split': [10, 15, 20], 
    'min_samples_leaf': [4, 6, 8], 
    'max_features': [0.5, 0.7, 'sqrt', 'log2'] 
}


# Initialize the RandomForestRegressor
rf = RandomForestRegressor(random_state=42)

# Setup RandomizedSearchCV
rf_random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist_rf, n_iter=200, 
                                      cv=5, verbose=2, random_state=42, n_jobs=-1)

# Fit RandomizedSearchCV
rf_random_search.fit(X_train, y_train)

# Print the best parameters
print("Best Parameters:", rf_random_search.best_params_)
print("Best Score:", rf_random_search.best_score_)

# Evaluate the best model from random search
enhanced_rf_model = RandomForestRegressor(**rf_random_search.best_params_)
evaluate_regression_model(enhanced_rf_model, X_train, y_train, X_test, y_test, "Enhanced Tuned Random Forest")

In [ ]:
evaluate_regression_model_with_cross_validation(enhanced_rf_model, X, y, "Cross Validation Enhanced Tuned Random Forest")

###### The Enhanced Tuned Random Forest model, after hyperparameter optimization with best parameters (`n_estimators`: 300, `min_samples_split`: 10, `min_samples_leaf`: 4, `max_features`: 0.5, `max_depth`: 20), shows an improved balance in performance. The training R² score is 79.47%, and the test R² score increases to 68.50%, indicating enhanced predictive accuracy on unseen data and a more effective control over overfitting compared to the original Random Forest model.

###### Error metrics for the enhanced model exhibit a training Mean Squared Error (MSE) of 0.1162 and a Mean Absolute Error (MAE) of 0.2203. On the test dataset, these metrics improve to an MSE of 0.1698 and an MAE of 0.2872, reflecting tighter prediction errors on new data. This improvement suggests that the hyperparameter tuning has positively impacted the model's generalization ability.

###### Cross-validation results further validate the enhancements, with an average test MSE of 0.2317, indicating a reduction in prediction error variability across folds compared to the original model. The MSE scores per fold range more narrowly (from 0.1962 to 0.2661), which underscores a more consistent performance across different data segments.

###### Compared to the original Random Forest model, the Enhanced Tuned version demonstrates a noteworthy improvement in mitigating overfitting, as evidenced by a closer alignment between training and testing performance and more stable cross-validation outcomes. This suggests that the hyperparameter tuning, particularly adjustments to the number of estimators, max depth, and sample split criteria, has effectively enhanced the model's accuracy and robustness.

###### In conclusion, the Enhanced Tuned Random Forest model represents a significant step forward in optimizing performance for regression tasks, offering a more reliable and generalizable model that better addresses the challenges of overfitting while maintaining strong predictive capabilities.


#### Enhanced Gradient Boosting Regressor

In [ ]:
# Enhanced Tuned Gradient Boosting Regressor
param_grid_gb = {
    'n_estimators': [150, 300],
    'learning_rate': [0.05, 0.1, 0.15, 0.2],
    'max_depth': [9, 11, 13, 15],
    'subsample': [1.0, 1.2, 1.4],
    'max_features': ['log2'],
    'min_samples_split': [10, 12],
    'min_samples_leaf': [4, 6, 8]
}

# Create a GridSearchCV object
grid_gb = GridSearchCV(estimator=GradientBoostingRegressor(), param_grid=param_grid_gb, cv=3, n_jobs=-1, verbose=2)

# Fit the model
grid_gb.fit(X_train, y_train)

# Print the best score and parameters
print("Best Score:", -grid_gb.best_score_)
print("Best Parameters:", grid_gb.best_params_)

# Create and evaluate the enhanced model with the best parameters
enhanced_gb_model = GradientBoostingRegressor(**grid_gb.best_params_)
enhanced_gb_model = evaluate_regression_model(
    enhanced_gb_model, X_train, y_train, X_test, y_test, "Enhanced Tuned Gradient Boosting"
)

In [ ]:
# K-Fold Cross Validation
evaluate_regression_model_with_cross_validation(enhanced_gb_model, X, y, "Cross Validation Enhanced Tuned Gradient Boosting") 

###### The Enhanced Tuned Gradient Boosting model, optimized with parameters (`learning_rate`: 0.05, `max_depth`: 11, `max_features`: 'log2', `min_samples_leaf`: 8, `min_samples_split`: 12, `n_estimators`: 150, `subsample`: 1.0), exhibits notable improvements over the original Gradient Boosting model. The training R² score significantly increases to 88.64%, suggesting a substantial capability to explain the variance in the target variable. The test R² score also sees an uplift to 69.94%, reflecting enhanced model accuracy on unseen data and a reduction in overfitting.

###### Error metrics show a promising decrease with a training Mean Squared Error (MSE) of 0.0643 and a Mean Absolute Error (MAE) of 0.1681. The test data metrics also improve, with an MSE of 0.1620 and an MAE of 0.2788, indicating more accurate predictions on new data compared to the original Gradient Boosting model.

###### Cross-validation analysis yields an average test MSE of 0.2353, with scores per fold ranging from 0.2007 to 0.2645. This consistency across folds demonstrates the model's improved generalization ability, showing a reliable performance across different subsets of data.

###### Compared to the original Gradient Boosting model, the Enhanced Tuned version achieves superior performance metrics, including higher R² scores and lower error rates, both in training and testing phases. The hyperparameter tuning, especially the adjustments in learning rate, max depth, and the incorporation of subsampling, has effectively addressed previous limitations, enhancing the model's predictive power and robustness against overfitting.

###### In summary, the Enhanced Tuned Gradient Boosting model represents a significant advancement in model optimization for regression tasks, providing a powerful and well-balanced approach that significantly reduces prediction errors while offering reliable performance across varied datasets.


#### Enhanced XGBoost Regressor

In [ ]:
# Define the parameter grid
param_dist_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],  # Decreased max depth
    'learning_rate': [0.05, 0.1, 0.15],
    'reg_alpha': [0.1, 0.5, 1],  # Increased L1 regularization
    'reg_lambda': [0.1, 0.5, 1],  # Increased L2 regularization
    'min_child_weight': [3, 5, 7],  # Increased
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8]
}


# Initialize the XGBRegressor
xgb = XGBRegressor(random_state=42)

# Setup RandomizedSearchCV
xgb_random_search = RandomizedSearchCV(estimator=xgb, param_distributions=param_dist_xgb, n_iter=200, cv=5, verbose=2, random_state=42, n_jobs=-1)

# Fit RandomizedSearchCV
xgb_random_search.fit(X_train, y_train)

# Print the best score and parameters
print("Best Score:", -xgb_random_search.best_score_)
print("Best Parameters:", xgb_random_search.best_params_)

# Create and evaluate the enhanced model with the best parameters
enhanced_xgb_model = XGBRegressor(**xgb_random_search.best_params_)
enhanced_xgb_model = evaluate_regression_model(enhanced_xgb_model, X_train, y_train, X_test, y_test, "Enhanced Tuned XGBoost")

In [ ]:
# K-Fold Cross Validation
evaluate_regression_model_with_cross_validation(enhanced_xgb_model, X, y, "Cross Validation Enhanced Tuned XGBoost")

###### The Enhanced Tuned XGBoost model, after optimization with the best parameters (`subsample`: 0.8, `reg_lambda`: 1, `reg_alpha`: 1, `n_estimators`: 100, `min_child_weight`: 5, `max_depth`: 5, `learning_rate`: 0.15, `colsample_bytree`: 0.8), showcases improved modeling efficiency. The training R² score of 77.43% indicates a robust ability to explain variance, while the test R² score increases to 66.41%, suggesting a commendable predictive accuracy on unseen data and a notable effort in mitigating overfitting compared to the original XGBoost model.

###### The model's error metrics on the training set include an MSE of 0.1277 and an MAE of 0.2496, which reveal precise prediction capability. On the test dataset, these metrics slightly increase to an MSE of 0.1811 and an MAE of 0.3051, demonstrating a reasonable accuracy in predictions on new data and indicating an improvement in generalization compared to the pre-tuning phase.

###### Cross-validation with five folds presents an average test MSE of 0.2347, with individual fold scores ranging from 0.1984 to 0.2685. This cross-validation performance denotes the model's stable and reliable predictive ability across various data segments, illustrating enhanced generalization post-tuning.

###### Compared to the initial XGBoost model, the Enhanced Tuned version shows a balanced improvement in reducing overfitting, as evidenced by a tighter gap between training and testing accuracy, alongside more consistent cross-validation outcomes. The fine-tuning of hyperparameters, including regularization terms, subsampling rates, and tree-specific parameters, has successfully enhanced the model's predictive performance and robustness.

###### In conclusion, the Enhanced Tuned XGBoost model emerges as a significantly optimized solution for regression tasks, achieving a better balance between model complexity and generalization ability, which ensures more accurate and reliable predictions across diverse datasets.


#### Enhanced Light GBM Regressor

In [ ]:
# Defining the parameter grid
param_grid_light = {
    'n_estimators': [100, 500],
    'max_depth': [5, 10, 15],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 1.0],
    'reg_alpha': [0, 1, 10],  # L1 regularization
    'reg_lambda': [0, 1, 10],  # L2 regularization
    'num_leaves': [20, 40],
}

# Setup GridSearchCV
grid_search_light = GridSearchCV(estimator=LGBMRegressor(random_state=42), param_grid=param_grid_light, scoring='neg_mean_squared_error', cv=5, verbose=0, n_jobs=-1)

# Perform the search
grid_search_light.fit(X_train, y_train)

# Print the best score and parameters
print("Best Score:", -grid_search_light.best_score_)
print("Best Parameters:", grid_search_light.best_params_)

enhanced_light_model = LGBMRegressor(**grid_search_light.best_params_)

evaluate_regression_model(enhanced_light_model, X_train, y_train, X_test, y_test, "Enhanced LightGBM")

In [ ]:
# K-Fold Cross Validation
evaluate_regression_model_with_cross_validation(enhanced_light_model, X, y, "Cross Validation Enhanced LightGBM")

###### The Enhanced LightGBM model, optimized with best parameters (`colsample_bytree`: 0.6, `learning_rate`: 0.1, `max_depth`: 15, `n_estimators`: 100, `num_leaves`: 40, `reg_alpha`: 1, `reg_lambda`: 1, `subsample`: 0.6), presents a refined predictive performance. The training R² score of 78.31% underscores the model's strong capability to capture the variance of the target variable. The test R² score improves to 68.62%, indicating enhanced accuracy on unseen data and a successful reduction in overfitting compared to the original LightGBM model.

###### Error metrics for the enhanced model show a training Mean Squared Error (MSE) of 0.1228 and a Mean Absolute Error (MAE) of 0.2408, indicating high precision in training predictions. The test dataset experiences a slight rise in MSE to 0.1692 and MAE to 0.2939, suggesting an effective prediction accuracy on new data and demonstrating an improved generalization capability.

###### Cross-validation analysis yields an average test MSE of 0.2331, with scores per fold demonstrating a consistent performance across different data segments (ranging from 0.2020 to 0.2593). This performance affirms the model's robustness and adaptability, enhancing its reliability across varied datasets.

###### Compared to the baseline LightGBM model, the Enhanced Tuned version exhibits significant improvements, particularly in managing overfitting, as reflected by a closer alignment of training and testing scores and more uniform cross-validation outcomes. The strategic adjustment of hyperparameters such as tree complexity, learning rate, and regularization, alongside the model's structure modifications, has markedly optimized its predictive power.

###### In summary, the Enhanced LightGBM model stands out as a highly optimized and effective solution for regression challenges, ensuring more accurate and consistent predictions. Its enhanced balance between complexity and generalization ability marks a substantial advancement in regression modeling, showcasing its potential for diverse predictive tasks.


#### Enhanced CatBoost Regressor

In [ ]:
# Enhanced Tuned CatBoost Regressor
param_dist_cb = {
    'iterations': [500, 1000],
    'learning_rate': [0.01, 0.02, 0.03],
    'depth': [12, 24, 36],
    'l2_leaf_reg': [1, 5, 10, 15],
    'border_count': [32, 254],
    'bagging_temperature': [0, 5, 10]
}

random_search_cb = RandomizedSearchCV(
    CatBoostRegressor(verbose=False),
    param_distributions=param_dist_cb,
    n_iter=200,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Fit the model
random_search_cb.fit(X_train, y_train)

# Print the best score and parameters
print("Best Score:", -random_search_cb.best_score_)
print("Best Parameters:", random_search_cb.best_params_)

# Create and evaluate the enhanced model with the best parameters
enhanced_cb_model = CatBoostRegressor(**random_search_cb.best_params_, verbose=False)
enhanced_cb_model = evaluate_regression_model(
    enhanced_cb_model, X_train, y_train, X_test, y_test, "Enhanced Tuned CatBoost"
)

In [ ]:
# K-Fold Cross Validation
evaluate_regression_model_with_cross_validation(enhanced_cb_model, X, y, "Cross Validation Enhanced Tuned CatBoost")

###### The Enhanced Tuned CatBoost model, with its optimal parameters (`learning_rate`: 0.03, `l2_leaf_reg`: 15, `iterations`: 1000, `depth`: 12, `border_count`: 32, `bagging_temperature`: 5), achieves significant improvements in performance. The training R² score reaches an impressive 88.58%, showcasing the model's exceptional ability to explain the variance in the dataset. The test R² score also exhibits a strong result at 68.39%, indicating a substantial predictive accuracy on unseen data and a notable advancement in addressing overfitting issues.

###### Error metrics present a training Mean Squared Error (MSE) of 0.0647 and a Mean Absolute Error (MAE) of 0.1794, evidencing the model's high accuracy in training predictions. For the test data, these metrics see a slight increase to an MSE of 0.1704 and an MAE of 0.2902, still indicating a robust performance and effective generalization capability from the model.

###### Cross-validation results further confirm the model's enhanced performance, with an average test MSE of 0.2392, and scores per fold ranging from 0.2047 to 0.2723. This consistency in cross-validation suggests that the Enhanced Tuned CatBoost model performs reliably across different data subsets, enhancing its overall generalizability.

###### Compared to the initial CatBoost model, the Enhanced Tuned version shows remarkable improvements in predictive accuracy and overfitting control. The fine-tuning of learning rate, depth, and regularization parameters, alongside iterative adjustments, has significantly optimized the model's efficiency and adaptability.

###### In conclusion, the Enhanced Tuned CatBoost model represents a substantial leap forward in the domain of regression modeling. Its optimized configuration ensures a powerful balance between capturing complex patterns in the training data and maintaining high accuracy on new, unseen datasets, positioning it as a highly effective tool for predictive tasks.

#### Stacking Regressor with Tuned Models

In [ ]:
# Stacking Regressor
base_models = [
    ('rf', RandomForestRegressor(n_estimators=600, random_state=42, max_depth=11, min_samples_leaf=1, min_samples_split=3, max_features=0.35)),
    ('xgb', XGBRegressor(n_estimators=400, learning_rate=0.2, random_state=42, 
                         max_depth=5, min_child_weight=3, reg_alpha=0.15, reg_lambda=0.15, subsample=0.3, colsample_bytree=0.3)),
]
meta_model = LinearRegression()

train_predictions_for_stacking = []
for name, model in base_models:
    model.fit(X_train, y_train) 
    train_predictions = model.predict(X_train)
    train_predictions_for_stacking.append(train_predictions)

# Convert the list of train predictions to a NumPy array for stacking
train_predictions_for_stacking = np.array(train_predictions_for_stacking).T

stacked_model = StackingRegressor(estimators=base_models, final_estimator=meta_model)

# Train the stacked model on the base model predictions made on the train set
stacked_model.fit(train_predictions_for_stacking, y_train) 

# Make predictions on the train set using the stacked model to calculate training scores
stacked_train_predictions = stacked_model.predict(train_predictions_for_stacking)

# Calculate training scores
mse_train = mean_squared_error(y_train, stacked_train_predictions)
r2_train = r2_score(y_train, stacked_train_predictions)
mae_train = mean_absolute_error(y_train, stacked_train_predictions)

# Fit the base models to the training data
for name, model in base_models:
    model.fit(X_train, y_train)

# Make predictions using your base models on the testing data
base_model_predictions = []
for name, model in base_models:
    predictions = model.predict(X_test) 
    base_model_predictions.append(predictions)

# Convert the list of base model predictions to a NumPy array
base_model_predictions = np.array(base_model_predictions).T

# Create the stacked ensemble model
stacked_model = StackingRegressor(estimators=base_models, final_estimator=meta_model)

# Train the stacked model on the base model predictions
stacked_model.fit(base_model_predictions, y_test)

# Make predictions using the stacked model
stacked_predictions = stacked_model.predict(base_model_predictions)

# Calculate Mean Squared Error (MSE)
mse_test = mean_squared_error(y_test, stacked_predictions)

# Calculate R-squared (R2) score
r2_test = r2_score(y_test, stacked_predictions)

# Calculate MAE for the stacked model predictions
mae_test = mean_absolute_error(y_test, stacked_predictions)

# Print the training scores
print('Train Mean Squared Error (MSE):', mse_train)
print('Train R-squared (R2) Score:', r2_train)
print(f"Train MAE for Stacked Model: {mae_train}")
# Print the results
print('Test Mean Squared Error (MSE):', mse_test)
print('Test R-squared (R2) Score:', r2_test)
print(f"Test MAE for Stacked Model: {mae_test}")


###### The Stacking Regressor, integrating optimized base models (Random Forest and XGBoost) with a meta-model (Linear Regression), demonstrates exceptional performance. Utilizing hyperparameters fine-tuned for each base model, this ensemble approach achieves a training R² score of 92.10%, highlighting its remarkable efficiency in capturing variance within the dataset. The test R² score also impresses at 88.04%, showcasing substantial predictive accuracy on unseen data, indicating an effective strategy in addressing overfitting.

###### The error metrics further underscore the stacked model's precision: a training Mean Squared Error (MSE) of 0.0447 and a Mean Absolute Error (MAE) of 0.1571 reflect the model's accuracy in training predictions. On the test dataset, these metrics modestly increase to an MSE of 0.0645 and an MAE of 0.1902, yet they remain indicative of robust performance and strong generalization capability.

###### This stacked ensemble model represents a strategic advancement over individual model performances, achieving lower prediction errors and higher R² scores. By leveraging the strengths of both Random Forest and XGBoost models, and harmonizing their predictions through a Linear Regression meta-model, it effectively minimizes bias and variance, leading to improved accuracy and reliability.

###### In summary, the Stacking Regressor model emerges as a highly optimized solution for complex regression tasks, ensuring precise and consistent predictions. Its success demonstrates the power of ensemble learning in enhancing predictive performance, making it a potent tool for tackling diverse predictive challenges with high accuracy and minimal overfitting.


### 2.4 Best Model

Among the models evaluated, the **Stacking Regressor** stands out as the most effective solution for the given regression tasks. It achieved a training **R² score of 92.10%** and a test **R² score of 88.04%**, indicating its superior ability to explain variance in the dataset and maintain high predictive accuracy on unseen data. The error metrics—training **MSE of 0.0447**, training **MAE of 0.1571**, test **MSE of 0.0645**, and test **MAE of 0.1902**—further demonstrate its precision and generalization capability.

The Stacking Regressor's strength lies in its ensemble approach, combining optimized base models (Random Forest and XGBoost) with a Linear Regression meta-model. This method effectively leverages the unique strengths of each base model, while the meta-model refines and enhances prediction accuracy by learning from the base models' predictions. Such a strategy addresses overfitting and reduces prediction errors, offering a balanced solution that surpasses the performance of individual models.

In comparison to the other models evaluated, including Enhanced Tuned versions of Random Forest, Gradient Boosting, XGBoost, LightGBM, and CatBoost, the Stacking Regressor not only achieved the highest R² scores but also maintained lower error rates, showcasing its robustness and adaptability across different datasets. Its ability to minimize bias and variance, leading to improved accuracy and reliability, positions it as the preferable choice for complex regression challenges.

Overall, the **Stacking Regressor model** represents a significant advancement in regression modeling, providing a potent and reliable tool for achieving precise and consistent predictions. Its optimized configuration ensures a powerful balance between capturing complex patterns in the training data and maintaining high accuracy on new, unseen datasets.
